# Dataset Filtering Summary

This notebook:
1. Loads the dataset from local checkpoint
2. Filters to keep only examples containing hard 'S' (s̪) sounds
3. Displays filtering statistics
4. Uploads the filtered dataset to HuggingFace Hub as a private repository

**Remember to:**
- Login to HuggingFace: `huggingface-cli login`
- Update the `repo_name` variable with your desired repository name

In [8]:
# Import necessary libraries
from datasets import load_from_disk, concatenate_datasets
import os
from pathlib import Path
from dotenv import load_dotenv

In [9]:
load_dotenv("../.env")

True

In [10]:
# Load dataset from chunked checkpoint
dataset_path = "/home/qklent/programming/speech_disorder_correction/mlm/data_preparation/data/prod_checkpoint"

def load_chunked_dataset(base_path, split="train"):
    """Load all chunks from a chunked dataset and concatenate them."""
    split_path = Path(base_path) / split
    chunk_dirs = sorted([d for d in split_path.iterdir() if d.is_dir() and d.name.startswith("chunk_")])
    
    print(f"Found {len(chunk_dirs)} chunks for {split} split")
    
    datasets = []
    for chunk_dir in chunk_dirs:
        print(f"Loading {chunk_dir.name}...")
        chunk_dataset = load_from_disk(str(chunk_dir))
        datasets.append(chunk_dataset)
    
    # Concatenate all chunks
    combined = concatenate_datasets(datasets)
    print(f"Combined {split} dataset: {len(combined)} examples")
    return combined

# Load train and validation splits
print("Loading train split...")
train_dataset = load_chunked_dataset(dataset_path, "train")

print("\nLoading validation split...")
val_dataset = load_chunked_dataset(dataset_path, "validation")

# Create DatasetDict
from datasets import DatasetDict
dataset = DatasetDict({
    "train": train_dataset,
    "validation": val_dataset
})

print(f"\nOriginal dataset: {dataset}")
print(f"Total examples - Train: {len(dataset['train'])}, Validation: {len(dataset['validation'])}")

Loading train split...
Found 10 chunks for train split
Loading chunk_0000...
Loading chunk_0001...
Loading chunk_0002...
Loading chunk_0003...
Loading chunk_0004...
Loading chunk_0005...
Loading chunk_0006...
Loading chunk_0007...
Loading chunk_0008...
Loading chunk_0009...
Combined train dataset: 91976 examples

Loading validation split...
Found 1 chunks for validation split
Loading chunk_0000...
Combined validation dataset: 4841 examples

Original dataset: DatasetDict({
    train: Dataset({
        features: ['audio', 'text', 'text_description', 'voice_name', 'phoneme_timestamps', 'processed', 'num_phonemes', 'error'],
        num_rows: 91976
    })
    validation: Dataset({
        features: ['audio', 'text', 'text_description', 'voice_name', 'phoneme_timestamps', 'processed', 'num_phonemes', 'error'],
        num_rows: 4841
    })
})
Total examples - Train: 91976, Validation: 4841


In [11]:
# Check the dataset structure
print(dataset["train"][0].keys())

dict_keys(['audio', 'text', 'text_description', 'voice_name', 'phoneme_timestamps', 'processed', 'num_phonemes', 'error'])


In [12]:
# Function to check if hard 'S' (s̪) is present
def has_hard_s(example):
    """
    Check if the example contains hard 'S' sound (s̪ phoneme).
    Returns True if at least one hard 'S' is present, False otherwise.
    """
    if "phoneme_timestamps" not in example or example["phoneme_timestamps"] is None:
        return False

    for phoneme_info in example["phoneme_timestamps"]:
        # Hard 'S' is represented as 's̪' in IPA (with diacritic)
        if phoneme_info["phoneme"] == "s̪":
            return True

    return False

In [13]:
# Filter dataset to keep only examples with hard 'S'
print("Filtering dataset to keep only examples with hard 'S' (s̪)...")
filtered_dataset = dataset.filter(has_hard_s)
print(f"\nFiltered dataset: {filtered_dataset}")
print(f"Filtered examples - Train: {len(filtered_dataset['train'])}, Validation: {len(filtered_dataset['validation'])}")

Filtering dataset to keep only examples with hard 'S' (s̪)...


Filter:   0%|          | 0/91976 [00:00<?, ? examples/s]

ValueError: No valid stream found in input file. Is -1 of the desired media type?

In [ ]:
# Display statistics
original_train = len(dataset["train"])
original_val = len(dataset["validation"])
filtered_train = len(filtered_dataset["train"])
filtered_val = len(filtered_dataset["validation"])

print("\n=== Filtering Statistics ===")
print(f"Train set: {filtered_train}/{original_train} ({filtered_train / original_train * 100:.1f}% kept)")
print(f"Validation set: {filtered_val}/{original_val} ({filtered_val / original_val * 100:.1f}% kept)")
print(
    f"Total: {filtered_train + filtered_val}/{original_train + original_val} ({(filtered_train + filtered_val) / (original_train + original_val) * 100:.1f}% kept)"
)

In [ ]:
# Verify an example from filtered dataset
example = filtered_dataset["train"][0]
hard_s_count = sum(1 for p in example["phoneme_timestamps"] if p["phoneme"] == "s̪")
print(f"Example text: {example['text']}")
print(f"Number of hard 'S' sounds: {hard_s_count}")
print(f"Voice: {example['voice_name']}")

In [ ]:
# Push to HuggingFace Hub (private)
# Make sure to login first: huggingface-cli login
# Or use: from huggingface_hub import login; login()

repo_name = "qklent/tonebooks-mfa-phonemes"  # CHANGE THIS to your desired repo name
print(f"Pushing filtered dataset to HuggingFace Hub: {repo_name}")
print("Note: This will be a private dataset")

filtered_dataset.push_to_hub(
    repo_name,
    private=True,
    token=os.getenv("HF_TOKEN"), 
)